# ДЗ 6. Кластеризация: DBSCAN vs KMeans
**Выполнил:** Хритинин Д. А.

**Задание:**
1. Запустить `DBSCAN` на датасете Wine (`eps=1.5, min_samples=5`).
2. Посчитать `silhouette_score` и `adjusted_rand_score`.
3. Сравнить с KMeans. Какой алгоритм лучше справился с поиском истинных сортов? Почему?

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Загрузка и масштабирование данных
Оба алгоритма опираются на расстояния, поэтому признаки стандартизируем.

In [ ]:
X, y = load_wine(return_X_y=True)
print("Размер:", X.shape, "| сорта:", np.bincount(y))
X_s = StandardScaler().fit_transform(X)

## 1–2. DBSCAN (eps=1.5, min_samples=5) и метрики

In [ ]:
db_labels = DBSCAN(eps=1.5, min_samples=5).fit_predict(X_s)

n_clusters = len(set(db_labels) - {-1})
n_noise = (db_labels == -1).sum()
print("Кластеров:", n_clusters, "| шумовых точек:", n_noise, "из", len(X_s))

db_ari = adjusted_rand_score(y, db_labels)
print("ARI DBSCAN:", round(db_ari, 4))

if n_clusters >= 2:
    mask = db_labels != -1
    db_sil = silhouette_score(X_s[mask], db_labels[mask])
    print("Silhouette DBSCAN:", round(db_sil, 4))
else:
    db_sil = np.nan
    print("Silhouette DBSCAN: не вычисляется (меньше 2 кластеров)")

## 3. KMeans (k = 3, по числу сортов)

In [ ]:
km_labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_s)
km_sil = silhouette_score(X_s, km_labels)
km_ari = adjusted_rand_score(y, km_labels)
print("Silhouette KMeans:", round(km_sil, 4))
print("ARI KMeans:", round(km_ari, 4))

### Почему DBSCAN не нашёл кластеры: подбор eps

In [ ]:
rows = []
for eps in [1.5, 2.0, 2.3, 2.5, 3.0]:
    l = DBSCAN(eps=eps, min_samples=5).fit_predict(X_s)
    rows.append({"eps": eps, "кластеров": len(set(l) - {-1}),
                 "шум": (l == -1).sum(), "ARI": round(adjusted_rand_score(y, l), 3)})
pd.DataFrame(rows)

### Визуализация (проекция на 2 главные компоненты)

In [ ]:
X_2d = PCA(n_components=2).fit_transform(X_s)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, labels, title in [(axes[0], y, "Истинные сорта"),
                          (axes[1], km_labels, f"KMeans (ARI={km_ari:.2f})"),
                          (axes[2], db_labels, f"DBSCAN eps=1.5 (ARI={db_ari:.2f})")]:
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap="viridis", s=20)
    ax.set_title(title); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.show()

In [ ]:
pd.DataFrame({
    "Алгоритм": ["DBSCAN (eps=1.5)", "KMeans (k=3)"],
    "Кластеров": [n_clusters, 3],
    "Silhouette": [db_sil, round(km_sil, 3)],
    "ARI": [round(db_ari, 3), round(km_ari, 3)],
})

## Выводы
| Алгоритм | Кластеров | Silhouette | ARI |
|---|---|---|---|
| DBSCAN (eps=1.5) | 0 (все 178 точек — шум) | не вычисляется | 0.00 |
| KMeans (k=3) | 3 | ≈ 0.285 | ≈ 0.897 |

**Лучше справился KMeans**: ARI ≈ 0.90, т.е. найденные кластеры почти совпадают с истинными сортами.

**Почему DBSCAN провалился:**
- В датасете 13 признаков, и после стандартизации расстояния между точками в таком пространстве получаются большими («проклятие размерности»). При радиусе `eps=1.5` ни у одной точки нет 5 соседей, поэтому ни одна не становится «ядровой» и все точки помечаются как шум.
- Даже при подборе eps DBSCAN даёт максимум ARI ≈ 0.38 (eps≈2.3): при меньших значениях он дробит данные на мелкие группы, при больших сливает всё в один кластер. Плотность между сортами недостаточно падает, чтобы алгоритм на основе плотности их разделил.

**Почему KMeans хорошо подходит:** сорта вина — это три компактных, примерно сферических облака похожего размера, а это ровно то предположение, на котором построен KMeans. К тому же число кластеров (3) известно заранее. Преимущества DBSCAN (кластеры произвольной формы, автоматическое число кластеров, поиск выбросов) на этих данных не нужны.